# MuseTalk Lip-Sync on Google Colab

Free GPU-powered lip-sync for your videos. Takes an existing video + a new audio track and syncs the lips to the audio — HeyGen-style, fully open-source (MIT).

**Before you start:** in Colab, go to `Runtime → Change runtime type` and select **GPU** (T4 is free and fine).

## Step 1 — Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU — enable it via Runtime → Change runtime type → GPU")

## Step 2 — Clone MuseTalk & install dependencies

In [ ]:
%cd /content
!git clone -q https://github.com/TMElyralab/MuseTalk.git
%cd /content/MuseTalk
!pip install -q -r requirements.txt

## Step 3 — Download model weights

~4GB, one-time download. Takes a few minutes.

In [ ]:
%cd /content/MuseTalk
!bash download_weights.sh

## Step 4 — Upload your video and audio

- **Video**: talking-head footage (MP4, face clearly visible)
- **Audio**: narration (WAV/MP3) — e.g. generated with the Sopro voice-cloning pipeline

In [ ]:
from google.colab import files
import shutil

print("⬆️ Upload your VIDEO file (mp4):")
video_upload = files.upload()
print("⬆️ Upload your AUDIO file (wav/mp3):")
audio_upload = files.upload()

video_path = "/content/MuseTalk/input_video.mp4"
audio_path = "/content/MuseTalk/input_audio.wav"

shutil.move(list(video_upload.keys())[0], video_path)
shutil.move(list(audio_upload.keys())[0], audio_path)
print(f"✅ Video: {video_path}")
print(f"✅ Audio: {audio_path}")

## Step 5 — Run lip-sync

On a T4 GPU: roughly 1–2 min per 10 seconds of video.

In [ ]:
with open("/content/MuseTalk/configs/inference/myvideo.yaml", "w") as f:
    f.write('task_0:\n video_path: "/content/MuseTalk/input_video.mp4"\n audio_path: "/content/MuseTalk/input_audio.wav"\n')
print("Config written.")

In [ ]:
%cd /content/MuseTalk
!python scripts/inference.py \
  --inference_config configs/inference/myvideo.yaml \
  --result_dir ./results \
  --unet_model_path ./models/musetalkV15/unet.pth \
  --unet_config ./models/musetalkV15/musetalk.json \
  --whisper_dir ./models/whisper

## Step 6 — Preview & download

In [ ]:
import glob
from IPython.display import Video
from google.colab import files

results = glob.glob("/content/MuseTalk/results/**/*.mp4", recursive=True)
print("Results:", results)
if results:
    display(Video(results[0], width=480))
    files.download(results[0])
else:
    print("⚠️ No output found — check the inference logs above.")